# Social Agent Negotiation — GRPO Training


In [ ]:
!pip install unsloth trl datasets transformers accelerate -q


In [ ]:
import torch, requests, json, re, time, matplotlib
from google.colab import userdata
from unsloth import FastLanguageModel
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

SPACE_URL = "https://bharath-1608-social-agent-negotiation-v1.hf.space"
TRAIN_TASKS = ["single-round-consensus", "adversarial-information", "opioid-overdose"]
EPISODES_PER_TASK = 20
N_EPOCHS = 3
GROUP_SIZE = 4


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    token=HF_TOKEN
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05
)
FastLanguageModel.for_inference(model)
print("Model ready")


In [ ]:
def rollout_episode(model, tokenizer, task_id) -> float:
    try:
        reset_resp = requests.post(f"{SPACE_URL}/reset", json={"task_id": task_id}, timeout=30).json()
        obs_a = reset_resp.get("obs_agent_a", {})
        obs_b = reset_resp.get("obs_agent_b", {})
    except Exception:
        return 0.05

    for turn in range(40):
        agent_id = "agent_a" if turn % 2 == 0 else "agent_b"
        obs = obs_a if agent_id == "agent_a" else obs_b
        
        prompt = f"Task: {obs.get(\"task_description\", \"\")}\nPrivate Info: {str(obs.get(\"private_information\", \"\"))[:500]}\nHistory: {str(obs.get(\"shared_conversation_history\", \"\"))[-800:]}\nAvailable Actions: {obs.get(\"available_actions\", [])}\nGenerate JSON action with action_type, content, reasoning."
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=150, pad_token_id=tokenizer.eos_token_id)
        out_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        action_payload = {"action_type": "share_information", "content": "Fallback", "reasoning": "Fallback"}
        try:
            match = re.search(r"\{.*\}", out_text, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                if parsed.get("action_type") in obs.get("available_actions", []):
                    action_payload = parsed
        except Exception:
            pass

        try:
            step_resp = requests.post(f"{SPACE_URL}/step", json={
                "agent_id": agent_id,
                "action": {
                    "agent_id": agent_id,
                    "action_type": action_payload.get("action_type", "share_information"),
                    "content": action_payload.get("content", ""),
                    "reasoning": action_payload.get("reasoning", "")
                }
            }, timeout=30)
            
            if step_resp.status_code == 422:
                continue
            
            step_data = step_resp.json()
            obs_a = step_data.get("obs_agent_a", obs_a)
            obs_b = step_data.get("obs_agent_b", obs_b)
            
            if step_data.get("done", False):
                if "episode_result" in step_data and step_data["episode_result"]:
                    return float(step_data["episode_result"].get("total_reward", 0.05))
                try:
                    state_resp = requests.get(f"{SPACE_URL}/state", timeout=30).json()
                    return float(state_resp.get("episode_reward", 0.05))
                except Exception:
                    return 0.05
                    
        except Exception:
            return 0.05
            
    try:
        state_resp = requests.get(f"{SPACE_URL}/state", timeout=30).json()
        return float(state_resp.get("episode_reward", 0.05))
    except Exception:
        return 0.05

print("Rollout function ready")


In [ ]:
rewards_log = {task: [] for task in TRAIN_TASKS}
all_rewards = []

for epoch in range(1, N_EPOCHS+1):
    print(f"=== EPOCH {epoch}/{N_EPOCHS} ===")
    for task_id in TRAIN_TASKS:
        print(f"  Task: {task_id}")
        for ep in range(1, EPISODES_PER_TASK+1):
            score = rollout_episode(model, tokenizer, task_id)
            rewards_log[task_id].append(score)
            all_rewards.append(score)
            if ep % 5 == 0:
                recent = rewards_log[task_id][-5:]
                print(f"    ep {ep:>2}/{EPISODES_PER_TASK}  score={score:.4f}  avg={sum(recent)/len(recent):.4f}")

print("Training complete")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#2196F3", "#F44336", "#4CAF50"]
for i, task_id in enumerate(TRAIN_TASKS):
    scores = rewards_log[task_id]
    episodes = list(range(1, len(scores)+1))
    if len(scores) >= 3:
        smoothed = np.convolve(scores, np.ones(3)/3, mode="valid")
    else:
        smoothed = scores
    ax.plot(episodes, scores, alpha=0.3, color=colors[i % len(colors)])
    ax.plot(range(len(episodes) - len(smoothed) + 1, len(episodes) + 1), smoothed, color=colors[i % len(colors)], linewidth=2.5, label=task_id)

ax.set_xlabel("Episode", fontsize=13)
ax.set_ylabel("Reward Score", fontsize=13)
ax.set_title("Social Agent Negotiation — Reward Curve (GRPO Training)", fontsize=15)
ax.legend(fontsize=11)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved reward_curve.png")


In [ ]:
model.save_pretrained("negotiation_lora")
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="negotiation_lora",
    repo_id="Bharath-1608/social-agent-negotiation-grpo",
    repo_type="model",
    token=HF_TOKEN
)
print("Model pushed to HuggingFace")
